In [2]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-29 19:10:27.863528897 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [4]:
v2.dot(dv)

np.float64(0.019730422395141473)

In [6]:
from ingest import load_faq_data

documents = load_faq_data()

In [7]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [8]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [9]:
scores = X.dot(v1)

In [10]:
scores[:10]

array([0.50192225, 0.26513018, 0.76294114, 0.4437853 , 0.26213059,
       0.40594135, 0.35743901, 0.56009992, 0.45960497, 0.24433363])

In [11]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float64(0.7629411421659094))

In [12]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [13]:
top5 = np.argsort(scores)[-5:]

In [14]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [15]:
scores[top5]

array([0.76294114, 0.75793714, 0.7192131 , 0.65363121, 0.56009992])

In [16]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411421659094
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371360784597
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131018586465
{'course': 'machine-learning-zoomcamp', 'section': 'General